# Render a Selected Person

Load a resolved person-tracking run, choose one logical person ID, inspect the planned framing, and render diagnostic and final vertical videos. YOLO and InsightFace are not run again.

## 1. Configuration

Choose a person after reviewing `03_analyze_tracks_and_people.ipynb`. Every render setting is explicit so this notebook can also run headlessly.

In [ ]:
from pathlib import Path

WORKING_DIRECTORY = Path.cwd().resolve()
PROJECT_ROOT = WORKING_DIRECTORY.parent if WORKING_DIRECTORY.name == "notebooks" else WORKING_DIRECTORY
if not globals().get("PERSON_TRACKER_MASTER", False):
    mode = "ANALYSIS"
mode = str(mode).upper()
if mode not in {"ANALYSIS", "PRODUCTION"}:
    raise ValueError("mode must be 'ANALYSIS' or 'PRODUCTION'")
# Set to a run path/name to override the master, environment, or runs/latest.json.
RUN_DIRECTORY_OVERRIDE = None
if globals().get("PERSON_TRACKER_MASTER", False):
    RUN_DIRECTORY_OVERRIDE = globals().get("RUN_DIRECTORY")
# A master can supply TARGET_PERSON_ID. Standalone None uses notebook 03's selection.
if not globals().get("PERSON_TRACKER_MASTER", False):
    TARGET_PERSON_ID = None
else:
    TARGET_PERSON_ID = globals().get("TARGET_PERSON_ID")

# Vertical output and framing behavior
OUTPUT_SIZE = (1080, 1920)
ASPECT_RATIO = (9, 16)
HORIZONTAL_PADDING = 0.0
VERTICAL_PADDING = 0.05
LOOKAHEAD_SECONDS = 0.75
RESPONSE_TIME = 0.35
VISIBLE_EXPANSION = 0.10
TRANSITION_EXPANSION = 0.20
CONTAINMENT_PRIORITY = 0.50
MAX_POSITION_SPEED = 1.25
MAX_ZOOM_SPEED = 0.60

# Encoding
ENCODER_PRESET = "p6"
ENCODER_QUALITY = 18
PREVIEW_WIDTH = 1000

print(f"Configured run override: {RUN_DIRECTORY_OVERRIDE}")
print(f"Configured target override: {TARGET_PERSON_ID}")

## 2. Load and validate the resolved run

In [ ]:
import json
import sys
from collections import defaultdict

import cv2
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import HTML, Video, display

SRC_DIRECTORY = PROJECT_ROOT / "src"
if str(SRC_DIRECTORY) not in sys.path:
    sys.path.insert(0, str(SRC_DIRECTORY))
get_ipython().run_line_magic("load_ext", "person_tracker.notebook_magics")
print(f"Mode: {mode}")

from person_tracker.storage import load_observation_run, resolve_run_directory
from person_tracker.ui import load_selected_person

RUN_DIRECTORY = resolve_run_directory(
    PROJECT_ROOT / "runs", stage="resolved", explicit=RUN_DIRECTORY_OVERRIDE,
)
print(f"Resolved run directory: {RUN_DIRECTORY}")

required_files = [
    "manifest.json", "tracks.jsonl", "face_samples.jsonl",
    "face_embeddings.npy", "identities.jsonl",
]
missing_files = [name for name in required_files if not (RUN_DIRECTORY / name).exists()]
if missing_files:
    raise FileNotFoundError(f"Run is incomplete; missing: {missing_files}")

manifest, tracking_history, face_samples = load_observation_run(
    RUN_DIRECTORY, load_face_crops=False
)
identity_history = defaultdict(dict)
with (RUN_DIRECTORY / "identities.jsonl").open(encoding="utf-8") as stream:
    for line in stream:
        if not line.strip():
            continue
        record = json.loads(line)
        frame_no = int(record.pop("frame"))
        track_id = int(record.pop("track_id"))
        identity_history[frame_no][track_id] = record
identity_history = dict(identity_history)

INPUT_VIDEO = Path(manifest["source_video"])
if not INPUT_VIDEO.exists():
    raise FileNotFoundError(f"Source video is missing: {INPUT_VIDEO}")
fps = float(manifest["fps"])
start_frame = int(manifest["start_frame"])
end_frame = int(manifest["end_frame"])

available_person_ids = sorted({
    int(identity["person_id"])
    for identities in identity_history.values()
    for identity in identities.values()
})
SELECTION_PATH = RUN_DIRECTORY / "selected_person.json"
selection_was_loaded = TARGET_PERSON_ID is None
if selection_was_loaded:
    TARGET_PERSON_ID = load_selected_person(SELECTION_PATH)
if TARGET_PERSON_ID is None:
    raise ValueError(
        "No target person is selected. Use notebook 03 or set TARGET_PERSON_ID explicitly."
    )
TARGET_PERSON_ID = int(TARGET_PERSON_ID)
if TARGET_PERSON_ID not in available_person_ids:
    raise ValueError(
        f"Person {TARGET_PERSON_ID} is unavailable; choose one of {available_person_ids}"
    )

OUTPUT_DIRECTORY = PROJECT_ROOT / "output" / RUN_DIRECTORY.name / f"person-{TARGET_PERSON_ID}"
DIAGNOSTIC_VIDEO = OUTPUT_DIRECTORY / "tracking-and-framing.mp4"
FINAL_VIDEO = OUTPUT_DIRECTORY / "final-cropped.mp4"
BROWSER_PREVIEW = OUTPUT_DIRECTORY / "browser-preview.mp4"

target_frames_present = sorted(
    frame_no for frame_no, identities in identity_history.items()
    if any(int(identity["person_id"]) == TARGET_PERSON_ID for identity in identities.values())
)
print(f"Source: {INPUT_VIDEO}")
print(f"Run range: {start_frame}:{end_frame} ({(end_frame-start_frame)/fps:.2f}s)")
print(f"Available people: {available_person_ids}")
print(f"Person {TARGET_PERSON_ID} visible in {len(target_frames_present)} frames ({len(target_frames_present)/fps:.2f}s)")
print(f"Selection source: {'saved file' if selection_was_loaded else 'explicit override'}")
print(f"Output: {OUTPUT_DIRECTORY}")

## 3. Build the framing plan

The plan combines the selected person's raw tracking box, padded containment, disappearance/reappearance transitions, lookahead, and speed-limited camera motion.

In [ ]:
from person_tracker.framing import (
    SmoothContainmentBox,
    build_lookahead_boxes,
    build_reappearance_transitions,
    build_smooth_boxes,
    build_target_frames,
)

target_frames = build_target_frames(
    tracking_history, identity_history, TARGET_PERSON_ID, start_frame, end_frame,
    aspect_ratio=ASPECT_RATIO,
    horizontal_padding=HORIZONTAL_PADDING,
    vertical_padding=VERTICAL_PADDING,
)
reappearance_transitions = build_reappearance_transitions(
    target_frames, start_frame, end_frame
)
lookahead_boxes = build_lookahead_boxes(
    target_frames, start_frame, end_frame, fps,
    aspect_ratio=ASPECT_RATIO, lookahead_seconds=LOOKAHEAD_SECONDS,
)
smooth_box = SmoothContainmentBox(
    fps=fps, aspect_ratio=ASPECT_RATIO, response_time=RESPONSE_TIME,
    visible_expansion=VISIBLE_EXPANSION,
    transition_expansion=TRANSITION_EXPANSION,
    containment_priority=CONTAINMENT_PRIORITY,
    max_position_speed=MAX_POSITION_SPEED,
    max_zoom_speed=MAX_ZOOM_SPEED,
)
smooth_boxes = build_smooth_boxes(
    target_frames, reappearance_transitions, start_frame, end_frame, smooth_box,
    lookahead_boxes=lookahead_boxes,
)
planned_frames = sum(box is not None for box in smooth_boxes.values())
print(f"Direct target frames: {len(target_frames):,}")
print(f"Frames with a planned crop: {planned_frames:,}")
print(f"Reappearance transitions: {len(reappearance_transitions):,}")

## 4. Inspect representative framing decisions

Green is the raw tracker box, magenta is the padded containment box, and cyan is the final smooth 9:16 crop.

In [ ]:
%%skip_if_mode PRODUCTION

from person_tracker.io import read_nth_frame

sample_indices = np.linspace(0, len(target_frames_present) - 1, num=min(6, len(target_frames_present)), dtype=int)
sample_frames = [target_frames_present[index] for index in sample_indices]
fig, axes = plt.subplots(len(sample_frames), 1, figsize=(16, 7 * len(sample_frames)), squeeze=False)
for ax, frame_no in zip(axes[:, 0], sample_frames):
    frame = read_nth_frame(INPUT_VIDEO, frame_no)
    target = target_frames.get(frame_no)
    if target:
        x1, y1, x2, y2 = target["track"]["bbox"]
        ax.add_patch(plt.Rectangle((x1, y1), x2-x1, y2-y1, fill=False, color="lime", linewidth=3))
        x1, y1, x2, y2 = target["container_box"]
        ax.add_patch(plt.Rectangle((x1, y1), x2-x1, y2-y1, fill=False, color="magenta", linewidth=3))
    smooth = smooth_boxes.get(frame_no)
    if smooth:
        x1, y1, x2, y2 = smooth
        ax.add_patch(plt.Rectangle((x1, y1), x2-x1, y2-y1, fill=False, color="cyan", linewidth=3))
    ax.imshow(frame)
    ax.set_title(f"Frame {frame_no} | {(frame_no-start_frame)/fps:.2f}s")
    ax.axis("off")
plt.tight_layout()
plt.show()

## 5. Render the diagnostic tracking and framing video

This is a separate stage so it can be skipped during automated runs when only the final crop is needed.

In [ ]:
%%skip_if_mode PRODUCTION

from tqdm.auto import tqdm
from person_tracker.drawing import (
    build_face_samples_by_track, get_last_face_confidence, put_text, put_tracking_label
)
from person_tracker.video import render_video

OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)
face_samples_by_track = build_face_samples_by_track(face_samples)

def draw_diagnostic_frame(frame_no, frame):
    target = target_frames.get(frame_no)
    smooth = smooth_boxes.get(frame_no)
    if target:
        track = target["track"]
        identity = target["identity"]
        track_id = int(track["track_id"])
        x1, y1, x2, y2 = map(int, track["bbox"])
        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 3)
        ox1, oy1, ox2, oy2 = map(int, target["container_box"])
        cv2.rectangle(frame, (ox1, oy1), (ox2, oy2), (255, 0, 255), 3)
        face_confidence, _ = get_last_face_confidence(face_samples_by_track, track_id, frame_no)
        face_label = "--" if face_confidence is None else round(face_confidence * 100)
        put_tracking_label(frame, x1, y1 - 8, [
            track_id, TARGET_PERSON_ID, round(float(identity.get("confidence", 0)) * 100), face_label
        ])
    if smooth:
        sx1, sy1, sx2, sy2 = map(int, smooth)
        cv2.rectangle(frame, (sx1, sy1), (sx2, sy2), (255, 255, 0), 4)
    status = "visible" if target else "not visible"
    put_text(frame, f"Frame {frame_no} | Person {TARGET_PERSON_ID}: {status}", 20, 40)
    return frame

diagnostic_progress = tqdm(total=end_frame-start_frame, desc="Diagnostic render", unit="frame")
def update_diagnostic_progress(done, total):
    diagnostic_progress.n = done
    diagnostic_progress.refresh()

diagnostic_result = render_video(
    INPUT_VIDEO, DIAGNOSTIC_VIDEO, draw_frame=draw_diagnostic_frame,
    start_frame=start_frame, end_frame=end_frame,
    preset=ENCODER_PRESET, quality=ENCODER_QUALITY,
    progress_callback=update_diagnostic_progress,
)
diagnostic_progress.close()
print(f"Saved diagnostic video: {diagnostic_result['path']}")

### View the diagnostic video

In [ ]:
%%skip_if_mode PRODUCTION

from person_tracker.io import create_browser_preview

preview_path = create_browser_preview(DIAGNOSTIC_VIDEO, BROWSER_PREVIEW, width=PREVIEW_WIDTH)
display(Video(filename=str(preview_path), embed=True, width=PREVIEW_WIDTH,
              html_attributes="controls playsinline preload='metadata'"))

## 6. Render the final vertical crop

In [ ]:
from tqdm.auto import tqdm
from person_tracker.framing import crop_frame
from person_tracker.video import render_video

OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)
last_crop = next((box for box in smooth_boxes.values() if box is not None), None)

def crop_selected_person(frame_no, frame):
    global last_crop
    crop = smooth_boxes.get(frame_no)
    if crop is not None:
        last_crop = crop
    if last_crop is None:
        return cv2.resize(frame, OUTPUT_SIZE, interpolation=cv2.INTER_AREA)
    return crop_frame(frame, last_crop, OUTPUT_SIZE)

final_progress = tqdm(total=end_frame-start_frame, desc="Final vertical render", unit="frame")
def update_final_progress(done, total):
    final_progress.n = done
    final_progress.refresh()

final_result = render_video(
    INPUT_VIDEO, FINAL_VIDEO, draw_frame=crop_selected_person,
    start_frame=start_frame, end_frame=end_frame, output_size=OUTPUT_SIZE,
    preset=ENCODER_PRESET, quality=ENCODER_QUALITY,
    progress_callback=update_final_progress,
)
final_progress.close()
print(f"Saved final video: {final_result['path']}")
print(f"Frames: {final_result['frames']:,}")
print(f"Duration: {final_result['duration_seconds']:.2f}s")

### View the final video

In [ ]:
%%skip_if_mode PRODUCTION

display(Video(filename=str(FINAL_VIDEO), embed=True, width=540,
              html_attributes="controls playsinline preload='metadata'"))